<a href="https://colab.research.google.com/github/cesartejido/TFM_Producto_Diabetes/blob/main/TFM_Producto_IA_Diabetes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TFM - Clasificación de pacientes diabéticos (Pima Indians Diabetes Dataset)**

## Autor: Julio César Tejido González
## Director: Cristian Rodríguez


# ***Bloque 1: Carga, EDA y preprocesamiento***

In [ ]:
# El primero paso, es  preparar el entorno cargando las librerias que se utilizarán más adelante.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy import stats

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42  # Fijación de la semilla de aleatoriedad (RANDOM_STATE) para garantizar la reproducibilidad de los resultados.

 ## **1. Carga de datos**

In [ ]:
# El segundo paso, imprescindible antes de ejecutar las siguientes linea de coódigo, es cargar el dataset "diabetes.csv" en los archivos de Colab.

df = pd.read_csv("diabetes.csv") # Leemos el archivo que subido al entorno y lo convertimos en un DataFrame.

print(f"Dimensiones del dataset: {df.shape}")
df.head()
# Se ven en las primeras cinco filas del dataset.
# En las columnas SkinThickness y Insulin podemos ver el valoros 0, esos ceros son exactamente los datos faltantes enmascarados que se deben tratar.

Ocho variables predictoras:
* Pregnancies (Número de embarazos): variable numérica entera de tipo discreto, expresada como un simple recuento sin unidad de medida. Un número elevado de embarazos se asocia a mayor riesgo de diabetes gestacional previa.
* Glucose (Glucosa plasmática): variable numérica entera de naturaleza continua, medida en miligramos por decilitro (mg/dL) a las dos horas de un test de tolerancia oral a la glucosa. En una persona sana el valor se mantiene por debajo de 140, mientras que entre 140 y 199 se considera prediabetes y a partir de 200 se cumple el criterio diagnóstico de diabetes.
* BloodPressure (Presión arterial diastólica): variable numérica entera de tipo continuo, expresada en milímetros de mercurio (mm Hg). Se considera normal por debajo de 80, y a partir de 90 ya se habla de hipertensión.
* SkinThickness (Espesor del pliegue cutáneo del tríceps): variable numérica entera de tipo continuo, medida en milímetros (mm). Es un indicador indirecto de la grasa subcutánea, y en mujeres adultas suele situarse aproximadamente entre 15 y 30, aunque varía bastante según la edad y la complexión de cada persona.
* Insulin (Insulina sérica): variable numérica entera de tipo continuo, expresada en microunidades por mililitro (µU/mL) medidas a las dos horas. En ayunas lo habitual son valores entre 2 y 25, mientras que tras una sobrecarga oral de glucosa el rango se amplía notablemente, moviéndose de forma orientativa entre 16 y 166.
* BMI (Índice de masa corporal): variable numérica decimal de tipo continuo, expresada en kilogramos por metro cuadrado (kg/m²). Entre 18,5 y 24,9 se considera normopeso, de 25 a 29,9 sobrepeso, y desde 30 en adelante se clasifica como obesidad, que es uno de los principales factores de riesgo de la diabetes tipo 2.
* DiabetesPedigreeFunction (Función de pedigrí diabético): variable numérica decimal de tipo continuo y adimensional, es decir, sin unidad de medida. Resume la carga genética familiar de diabetes, cuanto mayor es el valor, mayor es el peso de los antecedentes familiares en el riesgo de la paciente.
* Age (Edad): variable numérica entera de tipo discreto, expresada en años. El riesgo de diabetes tipo 2 aumenta de forma clara con la edad.

Variable objetivo (tarjet):
* Outcome (Diagnóstico de diabetes): variable binaria. El valor 0 indica ausencia de diabetes y el valor 1 indica presencia, según el criterio diagnóstico de la Organización Mundial de la Salud aplicado en el estudio original.



In [ ]:
df.info() # Los valores NaN no son reconocidos como faltantes porque están difrazados de ceros.
df.describe() # Estadísticos básicos.

## **2. Identificación de valores faltantes enmascarados como ceros**

En este dataset, un valor de 0 en Glucose, BloodPressure, SkinThickness, Insulin o BMI no es un valor clínico real (un paciente no puede tener 0 de glucosa o de presión arterial en vida), por lo que se trata de un dato faltante codificado como 0.

La columna Pregnancies sí admite legítimamente el valor 0.

In [ ]:
cols_con_ceros_invalidos = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

resumen_missing = pd.DataFrame({
    "n_ceros": [(df[c] == 0).sum() for c in cols_con_ceros_invalidos],
    "pct_missing": [round(100 * (df[c] == 0).sum() / len(df), 2) for c in cols_con_ceros_invalidos],
}, index=cols_con_ceros_invalidos)

print(resumen_missing)

# Sustituimos los ceros inválidos por NaN para tratarlos como missing real
df_clean = df.copy()
df_clean[cols_con_ceros_invalidos] = df_clean[cols_con_ceros_invalidos].replace(0, np.nan)

In [ ]:
# Patrón de datos faltantes: ¿se solapan entre variables?
solapamiento = (df_clean["Insulin"].isna() & df_clean["SkinThickness"].isna()).sum()
print(f"Pacientes sin Insulin: {df_clean['Insulin'].isna().sum()}")
print(f"Pacientes sin SkinThickness: {df_clean['SkinThickness'].isna().sum()}")
print(f"Pacientes sin ambas a la vez: {solapamiento}")

# ¿Coste de eliminar en lugar de imputar?
completos = df_clean.dropna().shape[0]
print(f"\nRegistros completos: {completos} de {len(df_clean)} "
      f"({100*completos/len(df_clean):.1f}%)")
print(f"Se perderia el {100*(1-completos/len(df_clean)):.1f}% de la muestra si se eliminaran.")

# ¿La ausencia de dato se relaciona con el diagnóstico? (mecanismo MCAR/MAR vs MNAR)
from scipy import stats
for var in ["Insulin", "SkinThickness"]:
    tabla = pd.crosstab(df_clean[var].isna(), df_clean["Outcome"])
    _, p, _, _ = stats.chi2_contingency(tabla)
    print(f"\n{var} - asociacion entre ausencia de dato y diagnostico: p = {p:.4f}")

* El solapamiento es del 100% entre los pacientes con valores nulos en Insulin y SkinThickess, por lo que podemos afirmar que hubo un subgrupo de pacientes al que directamente no se le realizó el protocolo completo de pruebas.
* Optar por eliminar las filas con datos faltantes en lugar de imputarlas, probocaría la perdida del 49% de la muestra, quedando con menos de 400 pacientes, y bajando el potencial estadistico drásticamente.
* Al comprobar si las pacientes con datos faltantes tienen distinta proporción de diabetes que las demás, podemos concluir que no hay diferencia estadísticamente significativa (p = 0,29 para insulina y p = 0,17 para el pliegue cutáneo). Esto es importantísimo porque significa que los datos no faltan porque la paciente estuviera enferma.



Las siguientes comparativas se han hecho con menos valores de los 768 totales, porque al haber convertido los ceros en NaN, las funciones estadísticas y los gráficos ignoran automáticamente esos valores faltantes y trabajan solo con los datos realmente disponibles en cada variable.

### **2.1. Variables predictoras frente a la variable objetivo**

In [ ]:
variables_clinicas = ["Glucose", "BMI", "Age", "Insulin", "DiabetesPedigreeFunction"]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, var in zip(axes, variables_clinicas):
    sns.boxplot(x="Outcome", y=var, data=df_clean, ax=ax)
    ax.set_title(var)
    ax.set_xlabel("Diabetes (0 = No, 1 = Sí)")
plt.tight_layout()
plt.savefig("fig05_boxplots_por_outcome.png", dpi=150)
plt.show()

In [ ]:
resultados = []
for var in variables_clinicas:
    grupo_sano = df_clean.loc[df_clean["Outcome"] == 0, var].dropna()
    grupo_diabetico = df_clean.loc[df_clean["Outcome"] == 1, var].dropna()
    _, p_valor = stats.mannwhitneyu(grupo_sano, grupo_diabetico)
    resultados.append({
        "Variable": var,
        "Mediana sin diabetes": round(grupo_sano.median(), 2),
        "Mediana con diabetes": round(grupo_diabetico.median(), 2),
        "p-valor": f"{p_valor:.2e}"
    })

tabla_contrastes = pd.DataFrame(resultados)
print(tabla_contrastes.to_string(index=False))

Las cinco variables muestran diferencias significativas entre grupos, y la más contundente con diferencia es la glucosa: la mediana pasa de 107 mg/dL en las pacientes sin diabetes a 140 en las diagnosticadas, con un p-valor del orden de 10⁻⁴⁰. Ese salto es enorme y era esperable, porque la glucemia es el criterio diagnóstico de la propia enfermedad.

### **2.2. Relaciones entre variables predictoras**

In [ ]:
pares_clinicos = [
    ("Glucose", "Insulin", "Resistencia a la insulina"),
    ("BMI", "SkinThickness", "Adiposidad corporal"),
    ("Age", "Pregnancies", "Historia obstétrica acumulada"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (x, y, titulo) in zip(axes, pares_clinicos):
    sns.scatterplot(x=x, y=y, hue="Outcome", data=df_clean, alpha=0.6, ax=ax)
    r = df_clean[[x, y]].corr().iloc[0, 1]
    ax.set_title(f"{titulo}\n(r = {r:.2f})")
plt.tight_layout()
plt.savefig("fig06_relaciones_entre_predictoras.png", dpi=150)
plt.show()

* Glucosa e insulina correlacionan a 0.58, refleja el patrón de resistencia a la insulina donde el páncreas segrega más hormona sin conseguir bajar la glucemia.
* IMC y espesor del pliegue cutáneo llegan a 0.65, la más alta de las tres, algo lógico porque las dos miden lo mismo por vías distintas.
* Edad y número de embarazos correlacionan a 0,54, una colinealidad moderada.

## **3. Análisis exploratorio de datos (EDA)**


El análisis exploratorio no es un trámite descriptivo previo al modelado, sino la fase donde se establece la **validez de los datos** sobre los que se sostendrá todo lo demás.

In [ ]:
plt.figure(figsize=(10, 5))
sns.heatmap(df_clean[cols_con_ceros_invalidos].isnull(), cbar=False, cmap="viridis")
plt.title("Mapa de valores faltantes por variable")
plt.tight_layout()
plt.savefig("fig01_mapa_missing.png", dpi=150)
plt.show()

El mapa revela que las ausencias de SkinThickness e Insulin coinciden en los mismos registros, afirmando que a un subgrupo de pacientes no se le aplicó el protocolo completo de pruebas.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flatten(), df.columns[:-1]):
    sns.histplot(df_clean[col].dropna(), kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig("fig02_distribuciones.png", dpi=150)
plt.show()

Insulin, DiabetesPedigreeFunction, Age y Pregnancies muestran una clara asimetría positiva, con una cola larga hacia la derecha. Esto no es un defecto de los datos sino una realidad clínica: la mayoría de pacientes tiene valores moderados y unas pocas presentan cifras muy elevadas.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df_clean.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matriz de correlación")
plt.tight_layout()
plt.savefig("fig03_correlacion.png", dpi=150)
plt.show()

Los tres valores más altos que se ven son precisamente los que tienen sentido fisiológico: BMI con SkinThickness (0,65), Glucose con Insulin (0,58) y Pregnancies con Age (0,54).

Ninguna correlación alcanza niveles preocupantes de colinealidad, que suelen situarse por encima de 0.8, así que no hay motivo para eliminar variables por redundancia. También destaca que Glucose es la variable más correlacionada con Outcome (0.49), anticipando su papel predominante en el modelo.

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x="Outcome", data=df_clean)
plt.title("Balance de clases (0 = sin diabetes, 1 = con diabetes)")
plt.savefig("fig04_balance_clases.png", dpi=150)
plt.show()

print(df_clean["Outcome"].value_counts(normalize=True).round(3))

El grafico anterior muestra 500 casos negativos frente a 268 positivos, es decir un 65% frente a un 35%.

 Es un desbalance moderado, no severo, pero suficiente para obligar a estratificar la partición para que ambos conjuntos mantengan la misma proporción, aconseja compensar el peso de las clases en los modelos, y sobre todo invalida la exactitud como métrica principal, y priorizar la sensibilidad y AUC.

## **4. Separación en train/test ANTES de imputar**

Este paso es crítico para evitar fuga de datos: la imputación se ajusta únicamente con el conjunto de entrenamiento y luego se aplica (transform) al test, nunca al revés.

En el apartado anterior, se hizo el análisis de ausencias sobre el conjunto completo, dentro de la exploración. Aquí lo repetimos usando solo el train, de modo que ningún criterio de imputación dependa de datos que el modelo no debería haber visto todavía.

In [ ]:
# Separamos variables predictoras (X) de la variable objetivo (y).
X = df_clean.drop(columns=["Outcome"])
y = df_clean["Outcome"]

# Partición 80/20 estratificada por la variable objetivo.
# Gracias a "stratify=y" ambos conjuntos mantienen la misma proporción de pacientes con y sin diabetes.
# El uso de "random_state" permite que la partición sea reproducible.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Proporción de clase positiva en train:", y_train.mean().round(3))
print("Proporción de clase positiva en test:", y_test.mean().round(3))

In [ ]:
# Comprobamos sobre el train el porcentaje de ausencias por variable,
# que es lo que justifica tratar unas con mediana y otras con KNN.
cols_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

missing_train = pd.DataFrame({
    "n_missing": [X_train[c].isna().sum() for c in cols_missing],
    "pct_missing": [round(100 * X_train[c].isna().mean(), 2) for c in cols_missing],
}, index=cols_missing)
print("Ausencias en el train:")
print(missing_train)

# Repetimos aquí el chi-cuadrado que en la exploración hicimos sobre el conjunto completo.
print("\nRelación entre ausencia de dato y diagnóstico (solo train):")
for var in ["Insulin", "SkinThickness"]:
    tabla = pd.crosstab(X_train[var].isna(), y_train)
    _, p, _, _ = stats.chi2_contingency(tabla)
    print(f"  {var}: p = {p:.3f}")

El conjunto de entrenamiento queda con 614 pacientes y el de prueba con 154. La proporción de clase positiva es casi idéntica en ambos (0,349 frente a 0,351), lo que confirma que la estratificación ha funcionado y que ninguno de los dos conjuntos ha quedado sesgado hacia una clase. La comprobación de ausencias sobre el train confirma además el mismo patrón que ya veíamos en la exploración, con Insulin y SkinThickness como únicas variables con missingness alto, así que la estrategia de imputación se sostiene sobre datos de entrenamiento.

## **5. Estrategia de imputación diferenciada por variable**

- Glucose, BloodPressure, BMI: missingness bajo (<5%) --> La imputación simple por **mediana** es coherente con este nivel de ausencia tan pequeño.
- SkinThickness (30%) e Insulin (49%): missingness alto --> Se opta por **KNNImputer** como estrategia de imputación. Se considera apropiada aquí porque estima el valor faltante a partir de las pacientes más similares en el resto de variables, lo que encaja con el patrón de ausencias detectado en el EDA. No se compara formalmente con otros métodos de imputación, así que se plantea como la opción razonada para este caso, no como la mejor en términos absolutos.

Todo esto se encapsulado en ColumnTransformer dentro de un Pipeline para que el ajuste ocurra solo sobre X_train.

In [ ]:
# Agrupamos las variables según su tratamiento de imputación.
vars_missing_bajo = ["Glucose", "BloodPressure", "BMI"]
vars_missing_alto = ["SkinThickness", "Insulin"]
vars_sin_missing = ["Pregnancies", "DiabetesPedigreeFunction", "Age"]

# La definimos como función para que cada llamada devuelva un pipeline NUEVO y sin ajustar.
# Así evitamos reutilizar un mismo objeto ya entrenado en la validación cruzada o el GridSearch,
# que aunque scikit-learn lo clona internamente, es más limpio y menos confuso hacerlo explícito.
def crear_preprocesador():
    # El KNN escala por dentro antes de medir distancias, si no la insulina (rango grande)
    # dominaría la búsqueda de vecinos. Recibe además Glucose, BMI y Age como contexto.
    knn_escalado = Pipeline(steps=[
        ("escalado_previo", StandardScaler()),
        ("knn", KNNImputer(n_neighbors=5))])

    preprocesador = ColumnTransformer(transformers=[
        ("imputacion_simple", SimpleImputer(strategy="median"), ["BloodPressure"]),
        ("imputacion_knn", knn_escalado, ["Glucose", "BMI", "Age", "SkinThickness", "Insulin"]),
        ("passthrough", "passthrough", ["Pregnancies", "DiabetesPedigreeFunction"])])

    return Pipeline(steps=[
        ("preprocesador", preprocesador),
        ("escalado", StandardScaler())])

# Para la comprobación de formas usamos una instancia recién creada, ajustada solo sobre train.
pipeline_preprocesamiento = crear_preprocesador()
X_train_procesado = pipeline_preprocesamiento.fit_transform(X_train)
X_test_procesado = pipeline_preprocesamiento.transform(X_test)

print("Forma final tras preprocesamiento (train):", X_train_procesado.shape)

Aquí se define todo el preprocesamiento en un solo objeto. Las variables con pocos huecos se rellenan con la mediana, y las que tienen más ausencias, SkinThickness e Insulin, se estiman con KNN a partir de pacientes parecidos. Para que esa búsqueda sea justa, el KNN escala por dentro las variables antes de medir distancias y usa Glucose, BMI y Age como contexto. Todo se ajusta solo con el train, de modo que el test queda intacto y no hay fuga de datos.

Ahora comprobamos cómo queda la insulina tras imputar, representándola en sus unidades reales (µU/mL), que es lo que tiene sentido clínico. Sobre datos escalados verías números sin interpretación médica. La imputación se hace con el mismo criterio que en el pipeline real, con el KNN escalando por dentro para buscar vecinos, y luego se muestra el resultado en la escala original. Trabajo solo sobre el train, que es donde el imputador aprende, para no tocar el test.

In [ ]:
# Para el gráfico queremos la insulina en unidades reales (µU/mL), pero imputando
# con el mismo criterio que el pipeline. Para ello escalamos de neuvo, y de esta forma el KNN calcula correctamente las
# distancias; después imputamos, y deshacemos el escalado para volver a la escala original.
cols_knn = ["Glucose", "BMI", "Age", "SkinThickness", "Insulin"]
X_knn = X_train[cols_knn].copy()

escalador_tmp = StandardScaler()
X_knn_escalado = escalador_tmp.fit_transform(X_knn)

X_knn_imputado_escalado = KNNImputer(n_neighbors=5).fit_transform(X_knn_escalado)

# inverse_transform devuelve los valores a sus unidades originales.
X_knn_imputado = escalador_tmp.inverse_transform(X_knn_imputado_escalado)
X_knn_imputado = pd.DataFrame(X_knn_imputado, columns=cols_knn, index=X_train.index)

# Distribución de Insulina antes (solo valores reales) y después (ya imputada).
insulina_antes = X_train["Insulin"].dropna()
insulina_despues = X_knn_imputado["Insulin"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
sns.histplot(insulina_antes, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title(f"Insulina ANTES de imputar\n(solo valores reales, n={insulina_antes.shape[0]})")
axes[0].set_xlabel("Insulina (µU/mL)")

sns.histplot(insulina_despues, kde=True, ax=axes[1], color="indianred")
axes[1].set_title(f"Insulina DESPUÉS de imputar con KNN\n(todos los pacientes, n={insulina_despues.shape[0]})")
axes[1].set_xlabel("Insulina (µU/mL)")

plt.tight_layout()
plt.savefig("fig07_insulina_antes_despues.png", dpi=150)
plt.show()

# Comparación numérica de la distribución.
print("ANTES  -> media:", round(insulina_antes.mean(),1),
      "| mediana:", round(insulina_antes.median(),1),
      "| desv. típica:", round(insulina_antes.std(),1))
print("DESPUÉS-> media:", round(insulina_despues.mean(),1),
      "| mediana:", round(insulina_despues.median(),1),
      "| desv. típica:", round(insulina_despues.std(),1))
print("Valores imputados:", X_train["Insulin"].isna().sum())

Es importante analizar el antes y el después de la imputación centrándonos en la insulina porque es, con diferencia, la variable con más datos faltantes (casi el 47% en el train).

Es el caso más exigente: si la imputación mantiene la distribución aquí, donde se rellenan 290 de 614 valores, el resto de variables, con porcentajes de ausencia mucho menores, quedan cubiertas por el mismo razonamiento. Además, es la variable cuya fiabilidad más conviene vigilar, ya que un volumen tan alto de valores estimados podría condicionar su peso posterior en el modelo.

El pico que se observa tras imputar refleja que el KNN concentra las estimaciones en la zona central de la distribución al rellenar 290 valores. Aun así es la mejor opción, ya que la imputación por mediana habría dado el mismo número a todas esas pacientes, produciendo un pico aún más pronunciado y anulando su variabilidad.


# ***Bloque 2: Modelado predictivo y evaluación clínica***

Partiendo del pipeline de preprocesamiento validado en el bloque anterior, en esta fase comparamos varios algoritmos de clasificación en igualdad de condiciones, seleccionamos y ajustamos el más adecuado, y evaluamos su rendimiento sobre el conjunto de test reservado. El preprocesamiento se integra como primer paso de cada modelo, de modo que la imputación y el escalado se reajustan dentro de cada partición de la validación cruzada, evitando cualquier fuga de datos.

## **6. Preparación del modelado**

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, RocCurveDisplay,
                             ConfusionMatrixDisplay, recall_score)

# Utilizamos validación cruzada estratificada para mantener la proporción de clases en cada partición.
# Es la elección adecuada para un dataset pequeño y con clases desbalanceadas.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

Definimos la estrategia de validación cruzada estratificada en 5 particiones, respetando de esta forma el desbalanceo de clases en cada iteración y aprovechando lo mejor posible el dataset de tamaño moderado.

In [ ]:
# Comprobamos como la validación cruzada estratificada mantiene el % de diabéticas estable en cada partición.

from sklearn.model_selection import KFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
kf  = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

prop_estrat = [round(y_train.iloc[test].mean(), 3) for _, test in skf.split(X_train, y_train)]
prop_normal = [round(y_train.iloc[test].mean(), 3) for _, test in kf.split(X_train)]

print("Proporción de diabéticas por partición:")
print("  Con estratificación:", prop_estrat)
print("  Sin estratificación:", prop_normal)

Antes de comparar modelos conviene fijar bien cómo vamos a medirlos. Usamos validación cruzada estratificada por dos motivos que se ven en los datos:

- El primero es que troceamos la muestra manteniendo en cada parte el mismo porcentaje de pacientes con diabetes, en torno al 35%, cosa que sin estratificar no se cumple y hace que algunas particiones queden desequilibradas. Esto supone un problema, porque si a un trozo le tocan por azar pocas diabéticas, el modelo se evalúa sobre un grupo que no representa bien la realidad, y la nota que sacas de ahí no es de fiar.

- El segundo es que evaluar con una única partición sería arriesgado con tan pocos pacientes. La validación cruzada parte los datos en cinco, entrena con cuatro partes y evalúa con la quinta, y va rotando hasta que cada paciente ha servido tanto para entrenar como para evaluar, en momentos distintos. Así usas toda la muestra para las dos cosas, sin desperdiciar nada.
En vez de fiarlo todo a una tirada, hace cinco y te da la media.

In [ ]:
# Segundo: ¿por qué CV y no una sola partición? Repetimos un hold-out
# simple con 10 semillas y vemos cuánto baila el resultado por puro azar.
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score

pipe_lr = Pipeline([("preprocesamiento", crear_preprocesador()),
                    ("clasificador", LogisticRegression(max_iter=1000,
                     class_weight="balanced", random_state=RANDOM_STATE))])

aucs = []
for semilla in range(10):
    xa, xb, ya, yb = train_test_split(X_train, y_train, test_size=0.2,
                                      stratify=y_train, random_state=semilla)
    pipe_lr.fit(xa, ya)
    aucs.append(roc_auc_score(yb, pipe_lr.predict_proba(xb)[:, 1]))

scores_cv = cross_val_score(pipe_lr, X_train, y_train, cv=skf, scoring="roc_auc")

print(f"Una sola partición: el AUC varía de {min(aucs):.3f} a {max(aucs):.3f} según el azar.")
print(f"Validación cruzada: AUC medio {scores_cv.mean():.3f} (variación de solo ±{scores_cv.std():.3f}).")

El contraste deja claro el problema de fiarlo todo a una sola partición: el AUC baila casi nueve centésimas según la suerte del reparto, de 0,802 a 0,888. La validación cruzada corrige eso promediando cinco evaluaciones, y por eso su AUC medio de 0,844 es un valor mucho más estable y creíble para comparar los modelos.

## **7. Entrenamiento y comparación de modelos**

In [ ]:
# Reutilizamos los mismos cinco algoritmos que planteamos en el apartado anterior,
# aplicando class_weight="balanced" en los que lo admiten, ya que el dataset
# tiene un desbalanceo real hacia la clase "no diabético" (aprox. 65/35).
modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "SVM (kernel RBF)": SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),  # KNN no admite class_weight, no tiene ese parámetro
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)  # tampoco lo admite
}

# Métricas de interés en un contexto clínico: además de accuracy y AUC,
# nos importa mucho el recall (sensibilidad), porque un falso negativo
# en diabetes es más grave que un falso positivo.
metricas = ["roc_auc", "accuracy", "recall", "precision", "f1"]

resultados = []

for nombre, modelo in modelos.items():
    pipe = Pipeline([
        ("preprocesamiento", crear_preprocesador()),
        ("clasificador", modelo)
    ])

    cv_resultado = cross_validate(
        pipe,
        X_train, y_train,
        cv=skf,
        scoring=metricas,
        n_jobs=-1
    )

    fila = {"Modelo": nombre}
    for m in metricas:
        media = cv_resultado[f"test_{m}"].mean()
        desviacion = cv_resultado[f"test_{m}"].std()
        fila[m] = f"{media:.3f} ± {desviacion:.3f}"
        fila[f"{m}_ord"] = media  # columna auxiliar solo para poder ordenar la tabla

    resultados.append(fila)

tabla_comparativa = pd.DataFrame(resultados).sort_values("roc_auc_ord", ascending=False)
tabla_comparativa = tabla_comparativa.drop(columns=[c for c in tabla_comparativa.columns if c.endswith("_ord")])
tabla_comparativa

En esta celda se entrena cada uno de los cinco algoritmos dentro del mismo pipeline de preprocesamiento, usando la validación cruzada estratificada de 5 particiones definida antes. Para cada modelo se calculan cinco métricas (ROC-AUC, accuracy, recall, precision y F1) y se muestra la media junto con la desviación típica entre particiones, ordenando la tabla por ROC-AUC de mayor a menor.

La Regresión Logística queda en primer lugar con un ROC-AUC de 0,844, seguida muy de cerca por el SVM con 0,836. Lo interesante no es solo el orden por AUC, sino que el SVM saca un recall notablemente más alto (0,766 frente a 0,701 de la Regresión Logística), aunque a costa de algo de precisión. Esto ya apunta a que ambos modelos son los principales candidatos.

KNN es el que peor se comporta en todas las métricas, con el AUC más bajo (0,817) y también la mayor variabilidad entre particiones, lo que sugiere que es el modelo menos estable de los cinco para este dataset.

In [ ]:
# Boxplot comparando la distribución del AUC entre los 5 folds para cada modelo,
# igual que hicimos antes con la Regresión Logística sola, pero ahora para los cinco.

aucs_por_modelo = {}
for nombre, modelo in modelos.items():
    pipe = Pipeline([
        ("preprocesamiento", crear_preprocesador()),
        ("clasificador", modelo)
    ])
    aucs_por_modelo[nombre] = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="roc_auc")

plt.figure(figsize=(9, 5))
plt.boxplot(aucs_por_modelo.values(), tick_labels=aucs_por_modelo.keys())
plt.ylabel("ROC-AUC")
plt.title("Comparación de modelos mediante validación cruzada (5 particiones)")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("fig08_comparacion_modelos_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

El gráfico confirma lo que ya se veía en la tabla: la Regresión Logística tiene la caja más estrecha y más alta de las cinco, lo que indica que su rendimiento es el más consistente entre particiones, sin apenas variación. El SVM y Random Forest muestran cajas más anchas, con un rango que llega a superponerse con la Regresión Logística en el extremo superior pero también baja más en el inferior. KNN es, de nuevo, el modelo con mayor dispersión y el que peor mediana presenta, reforzando la idea de que es el candidato menos fiable de los cinco para seguir adelante.

## **8. Optimización de hiperparámetros**

Antes de entrenar un modelo hay que decidir ciertos ajustes internos, llamados hiperparámetros, que no se aprenden de los datos sino que se fijan de antemano y condicionan cómo se comporta el algoritmo. GridSearchCV permite probar de forma automática varias combinaciones de estos ajustes y quedarse con la que mejor resultado da, en lugar de fijarlos a mano por intuición.

Se aplica el GridSearchCV a los dos modelos que mejor resultado dieron en la comparación anterior, la Regresión Logística y el SVM. Se prueban distintas combinaciones de hiperparámetros usando las mismas 5 particiones estratificadas de siempre, y se selecciona la combinación que maximiza el ROC-AUC medio.

En la Regresión Logística se ajustan el parámetro de regularización C y el tipo de penalización, mientras que en el SVM se ajustan C y gamma, que regula la flexibilidad del kernel RBF.

In [ ]:
# Rejilla de hiperparámetros para la Regresión Logística.
# Probamos distintos valores de regularización (C) y dos tipos de penalización.
# El solver "liblinear" admite tanto l1 como l2, por eso lo fijamos para toda la rejilla.
param_grid_lr = {
    "clasificador__C": [0.01, 0.1, 1, 10, 100],
    "clasificador__penalty": ["l1", "l2"],
    "clasificador__solver": ["liblinear"]
}

pipe_lr = Pipeline([
    ("preprocesamiento", crear_preprocesador()),
    ("clasificador", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
])

grid_lr = GridSearchCV(
    pipe_lr,
    param_grid=param_grid_lr,
    scoring="roc_auc",
    cv=skf,
    n_jobs=-1
)
grid_lr.fit(X_train, y_train)

print("Mejores hiperparámetros (Regresión Logística):", grid_lr.best_params_)
print(f"Mejor ROC-AUC en validación cruzada: {grid_lr.best_score_:.3f}")

In [ ]:
# Rejilla de hiperparámetros para el SVM.
# C controla la penalización por error, gamma la influencia de cada punto de entrenamiento.
param_grid_svm = {
    "clasificador__C": [0.1, 1, 10, 100],
    "clasificador__gamma": ["scale", 0.01, 0.1, 1],
    "clasificador__kernel": ["rbf"]
}

pipe_svm = Pipeline([
    ("preprocesamiento", crear_preprocesador()),
    ("clasificador", SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE))
])

grid_svm = GridSearchCV(
    pipe_svm,
    param_grid=param_grid_svm,
    scoring="roc_auc",
    cv=skf,
    n_jobs=-1
)
grid_svm.fit(X_train, y_train)

print("Mejores hiperparámetros (SVM):", grid_svm.best_params_)
print(f"Mejor ROC-AUC en validación cruzada: {grid_svm.best_score_:.3f}")

In [ ]:
# Comparativa final entre los dos modelos ya optimizados, incluyendo recall,
# ya que es la métrica que más nos importa desde el punto de vista clínico.

from sklearn.model_selection import cross_val_score

modelos_optimizados = {
    "Regresión Logística (optimizada)": grid_lr.best_estimator_,
    "SVM (optimizado)": grid_svm.best_estimator_
}

resumen_optimizacion = []
for nombre, modelo in modelos_optimizados.items():
    auc = cross_val_score(modelo, X_train, y_train, cv=skf, scoring="roc_auc").mean()
    recall = cross_val_score(modelo, X_train, y_train, cv=skf, scoring="recall").mean()
    resumen_optimizacion.append({"Modelo": nombre, "ROC-AUC": round(auc, 3), "Recall": round(recall, 3)})

tabla_optimizacion = pd.DataFrame(resumen_optimizacion)
tabla_optimizacion

La Regresión Logística obtiene su mejor resultado con C=0.1 y penalización l2, subiendo apenas de 0.844 a 0.845 de ROC-AUC. El SVM mejora algo más, de 0.836 a 0.848, con C=1 y gamma=0.01, quedando ligeramente por delante en esta métrica. Sin embargo, ambos modelos igualan su recall en 0.72, así que en la capacidad de detectar pacientes diabéticos no hay diferencia entre ellos.

El SVM tiene algo más de discriminación global, pero la Regresión Logística resulta más interpretable, algo relevante en un entorno hospitalario real. Por eso no se descarta ninguno de los dos todavía, y la decisión final se deja para el siguiente apartado, donde la matriz de confusión, la curva ROC y el ajuste del umbral clínico aportarán más criterio para elegir el modelo definitivo.

## **9. Evaluación completa: matriz de confusión y curva ROC**

Hasta ahora todos los números que hemos manejado venían de la validación cruzada, es decir, de datos que en algún momento formaron parte del entrenamiento, pero en este apartado entrenamos cada modelo con todo el conjunto de train y lo evaluamos contra el test, que no ha intervenido en ninguna decisión anterior.

Es la prueba real para ver si los modelos generalizan o solo se veían bien porque los estábamos midiendo con "trampa". Se sacan el informe de clasificación, las matrices de confusión y las curvas ROC de los dos candidatos, Regresión Logística y SVM.

In [ ]:
# El GridSearchCV ya reentrenó el mejor modelo sobre todo el train (refit=True por defecto),
# así que best_estimator_ está listo para predecir directamente, sin volver a ajustarlo.

y_pred_lr = grid_lr.best_estimator_.predict(X_test)
y_pred_svm = grid_svm.best_estimator_.predict(X_test)

y_prob_lr = grid_lr.best_estimator_.predict_proba(X_test)[:, 1]
y_prob_svm = grid_svm.best_estimator_.predict_proba(X_test)[:, 1]

print("Regresión Logística, conjunto de test:")
print(classification_report(y_test, y_pred_lr))

print("SVM, conjunto de test:")
print(classification_report(y_test, y_pred_svm))

 Ambos modelos tiene una accuracy muy similar, por lo que tenemos que dar especialmente importancia es a la clase 1 (pacientes diabéticos): la Regresión Logística presenta un recall de 0,70 y el SVM sube ligeramente a 0,72, y con precision muy parecida en los dos (0,58 y 0,59). Son resultados prácticamente empatados, con el SVM minimamente superior detectando los casos positivos.

In [ ]:
# Matrices de confusión de los dos modelos, una al lado de otra para compararlas facilmente.
#  TP FP
#  FN TN

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, ax=axes[0], colorbar=False)
axes[0].set_title("Regresión Logística")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_svm, ax=axes[1], colorbar=False)
axes[1].set_title("SVM")

plt.tight_layout()
plt.savefig("fig09_matrices_confusion.png", dpi=300, bbox_inches="tight")
plt.show()

Comparando las dos matrices de confusión, podemos ver que los dos modelos son casi identicos. La Regresión Logística acierta 38 pacientes diabéticos y se le escapan 16, el SVM acierta uno más (39) y falla 15. Un solo paciente de diferencia. Y en los falsos positivos coinciden exactamente 27 en los dos. Objetivamente, mirando solo esta matriz no encuentro ningún argumento fuerte para quedarme con uno de los dos modelos.

In [ ]:
# Curvas ROC de ambos modelos sobre el conjunto de test.

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)

auc_lr_test = roc_auc_score(y_test, y_prob_lr)
auc_svm_test = roc_auc_score(y_test, y_prob_svm)

plt.figure(figsize=(6.5, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Regresión Logística (AUC = {auc_lr_test:.3f})")
plt.plot(fpr_svm, tpr_svm, label=f"SVM (AUC = {auc_svm_test:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("Tasa de falsos positivos")
plt.ylabel("Tasa de verdaderos positivos (sensibilidad)")
plt.title("Curva ROC en el conjunto de test")
plt.legend()
plt.tight_layout()
plt.savefig("fig10_curva_roc_test.png", dpi=300, bbox_inches="tight")
plt.show()

Las dos curvas van prácticamente pegadas durante todo el recorrido, con el SVM ligeramente por encima en algunos tramos. El AUC de test confirma lo mismo que ya veíamos en la validación cruzada: 0,810 para la Regresión Logística y 0,814 para el SVM, una diferencia mínima que no permite hablar de un modelo claramente superior, aunque el SVM es ligeramente superir en TODAS las métricas.

## **10. Ajuste del umbral de decisión para reducir falsos negativos**

En un problema donde un falso negativo significa mandar a casa a una paciente diabética sin diagnosticar, tiene mucho más sentido bajar ese umbral y aceptar algunos falsos positivos de más a cambio de pillar a más pacientes reales.

El umbral de 0.5 que usa un modelo por defecto no viene de ninguna consideración médica, es solo el punto medio entre las dos clases. Para mejorar la detección, lo movemos a propósito buscando, para cada modelo, un umbral más bajo que priorice la sensibilidad, es decir, cazar al mayor número posible de pacientes diabéticas aunque eso suponga aceptar algunas falsas alarmas. Más adelante fijaremos ese objetivo de recall de forma justificada, viendo el coste que tiene cada nivel.

Para calcular ese umbral hay que tener cuidado con un detalle que puede pasar desapercibido, y es que no podemos usar el conjunto de test para decidirlo. El test tiene que quedar reservado únicamente para dar la cifra final de rendimiento, y si lo usamos también para elegir el umbral, estamos tomando una decisión con información que en teoría no deberíamos conocer todavía.

In [ ]:
# Para cada modelo, calculamos precision, recall y umbral asociado en todo el rango posible.

from sklearn.model_selection import cross_val_predict

# En vez de predict_proba directo sobre el modelo ya entrenado con todo el train,
# usamos cross_val_predict: para cada paciente del train, la probabilidad que le asigna
# viene de un modelo que NO lo vio durante ese fold. Así simulamos "datos nuevos"
# sin tocar el test en ningún momento.
prob_lr_train_cv = cross_val_predict(
    grid_lr.best_estimator_, X_train, y_train, cv=skf, method="predict_proba"
)[:, 1]

prob_svm_train_cv = cross_val_predict(
    grid_svm.best_estimator_, X_train, y_train, cv=skf, method="predict_proba"
)[:, 1]

# Gráfico de recall según el umbral, ahora calculado sobre el train (vía CV), no sobre el test.

precision_lr_cv, recall_lr_cv, umbrales_lr_cv = precision_recall_curve(y_train, prob_lr_train_cv)
precision_svm_cv, recall_svm_cv, umbrales_svm_cv = precision_recall_curve(y_train, prob_svm_train_cv)

plt.figure(figsize=(8, 5))
plt.plot(umbrales_lr_cv, recall_lr_cv[:-1], label="Recall - Regresión Logística")
plt.plot(umbrales_svm_cv, recall_svm_cv[:-1], label="Recall - SVM")
plt.axvline(0.5, linestyle="--", color="gray", label="Umbral por defecto (0.5)")
plt.xlabel("Umbral de decisión")
plt.ylabel("Recall (sensibilidad)")
plt.title("Cómo cae el recall según subimos el umbral (calculado sobre train)")
plt.legend()
plt.tight_layout()
plt.savefig("fig11_recall_vs_umbral.png", dpi=300, bbox_inches="tight")
plt.show()

En el grafico anterior podemos ver claramente la importancia del umbral, ya que con el 0.5 de siempre, la Regresión Logística baja bastante el recall, pero el SVM lo hace incluso más rápido, mostrando una caída más marcada al pasar el umbral por defecto. La línea azul (Regresión Logística) aguanta mejor la sensibilidad conforme subimos el umbral e comparación  con la naranja (SVM), que se desploma antes.

In [ ]:
# Antes de cambiar el umbrar, vamos a crear una función que, dado un vector de probabilidades, devuelve el umbral más alto
# que todavía alcanza un recall objetivo. La usamos tanto para explorar varios
# niveles como para fijar el definitivo.
def umbral_para_recall_objetivo(y_prob, y_real, objetivo):
    precision, recall, umbrales = precision_recall_curve(y_real, y_prob)
    candidatos = [(u, r) for u, r in zip(umbrales, recall[:-1]) if r >= objetivo]
    if not candidatos:
        return None
    return max(candidatos, key=lambda x: x[0])[0]

In [ ]:
# Análisis de sensibilidad: en lugar de fijar el 85% sin más, probamos varios objetivos
# de recall y vemos qué precisión y cuántos falsos positivos implica cada uno.
# Todo calculado sobre el train (vía CV), sin tocar el test.

objetivos = [0.75, 0.80, 0.85, 0.90, 0.95]
filas = []

for obj in objetivos:
    u = umbral_para_recall_objetivo(prob_lr_train_cv, y_train, obj)
    if u is None:
        continue
    y_pred_obj = (prob_lr_train_cv >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, y_pred_obj).ravel()
    recall_real = tp / (tp + fn)
    precision_real = tp / (tp + fp) if (tp + fp) > 0 else 0
    filas.append({
        "Recall objetivo": obj,
        "Umbral": round(u, 3),
        "Recall real": round(recall_real, 3),
        "Precisión": round(precision_real, 3),
        "Falsos negativos": fn,
        "Falsos positivos": fp,
    })

tabla_sensibilidad = pd.DataFrame(filas)
print("Coste de cada nivel de recall (Regresión Logística, sobre train):")
print(tabla_sensibilidad.to_string(index=False))

La elección del recall objetivo no es arbitraria, sale de mirar el coste de cada nivel. En un cribado de diabetes el error grave es el falso negativo, dejar sin detectar a una paciente enferma, mientras que un falso positivo solo supone una prueba de confirmación adicional. Por eso buscamos un recall alto, pero sin pasarnos, porque exigir demasiada sensibilidad dispara las falsas alarmas.

La tabla lo muestra con claridad. Pasar del 85% al 90% de recall apenas recupera unas pocas diabéticas más, los falsos negativos bajan de 32 a 21, pero a cambio los falsos positivos suben de 148 a 171 y la precisión cae por debajo del 0,53. Llegar al 95% ya es claramente ineficiente, 214 falsos positivos para el mismo grupo de pacientes. El 85% se sitúa justo antes de que esa penalización se dispare, manteniendo un buen nivel de detección con un número de falsas alarmas todavía asumible. Es el punto de equilibrio que adoptamos como criterio clínico.

In [ ]:
# Fijamos el 85% como objetivo, según lo justificado en el análisis de sensibilidad anterior.
objetivo_recall = 0.85

umbral_lr_ajustado = umbral_para_recall_objetivo(prob_lr_train_cv, y_train, objetivo_recall)
umbral_svm_ajustado = umbral_para_recall_objetivo(prob_svm_train_cv, y_train, objetivo_recall)

print(f"Umbral ajustado Regresión Logística (recall ≥ 85% en train): {umbral_lr_ajustado:.3f}")
print(f"Umbral ajustado SVM (recall ≥ 85% en train): {umbral_svm_ajustado:.3f}")

Con el objetivo del 85% ya justificado, la Regresión Logística alcanza ese nivel de detección con un umbral de 0.381 y el SVM con uno de 0.258. El SVM necesita bajar más el umbral para lograr la misma sensibilidad, lo que confirma lo que ya se intuía en la gráfica anterior, pierde recall más deprisa y por eso tiene que ser más permisivo. Ese umbral más bajo es justo el que acabará arrastrando más falsos positivos en el SVM, algo que veremos al comparar las matrices de confusión.

In [ ]:
# Aplicamos los nuevos umbrales y comparamos matrices de confusión: antes (0.5) y después.

y_pred_lr_ajustado = (y_prob_lr >= umbral_lr_ajustado).astype(int)
y_pred_svm_ajustado = (y_prob_svm >= umbral_svm_ajustado).astype(int)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, ax=axes[0, 0], colorbar=False)
axes[0, 0].set_title("Regresión Logística — umbral 0.5")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr_ajustado, ax=axes[0, 1], colorbar=False)
axes[0, 1].set_title(f"Regresión Logística — umbral {umbral_lr_ajustado:.2f}")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_svm, ax=axes[1, 0], colorbar=False)
axes[1, 0].set_title("SVM — umbral 0.5")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_svm_ajustado, ax=axes[1, 1], colorbar=False)
axes[1, 1].set_title(f"SVM — umbral {umbral_svm_ajustado:.2f}")

plt.tight_layout()
plt.savefig("fig12_matrices_confusion_umbral_ajustado.png", dpi=300, bbox_inches="tight")
plt.show()

print("Regresión Logística, umbral ajustado:")
print(classification_report(y_test, y_pred_lr_ajustado))

print("SVM, umbral ajustado:")
print(classification_report(y_test, y_pred_svm_ajustado))

El efecto de bajar el umbral se nota en los dos modelos. La Regresión Logística reduce sus falsos negativos de 15 a 7, y el SVM de 15 a 9. Es decir, ajustando el umbral pasamos por alto a menos pacientes diabéticas, que era justo el objetivo. Los dos suben sus falsos positivos hasta 36, así que en falsas alarmas quedan igualados. La diferencia está en el otro lado, la Regresión Logística detecta a dos diabéticas más que el SVM (47 frente a 45) pagando el mismo precio en falsos positivos. Para el mismo coste, caza más enfermas, y en un cribado eso es exactamente lo que buscamos.

Con el umbral ajustado, la Regresión Logística alcanza un recall de 0.87 en la clase diabética, por encima del 0.83 del SVM. Y lo hace además con mejor precisión (0.57 frente a 0.56) y un f1 más alto (0.69 frente a 0.67). Es decir, no solo detecta a más pacientes enfermas, sino que se equivoca menos al señalarlas. Sumado a que es un modelo bastante más fácil de interpretar para el personal clínico, la balanza se inclina con claridad hacia la Regresión Logística como modelo definitivo del trabajo.

## **11. Interpretabilidad del modelo definitivo**

Por último, realizaremos el más importante de la aplicación de modelos algoritmicos en entornos hospitalarios reales, que es poder explicar por qué predice lo que predice, ya que un médico no va a fiarse de una caja negra que le suelta un número, necesita entender qué variables está pesando el modelo y en qué dirección.

La forma más coherente es hacerlo en 2 niveles:
* Primero una visión global, qué variables pesan más en el modelo en conjunto, con dos métodos que se apoyan mutuamente: los coeficientes de la propia Regresión Logística y la importancia por permutación.
* Despues, una visión local con SHAP, que permite explicar casos concretos, por qué el modelo marcó a esta paciente en particular.

### **11.1. Visión global**

In [ ]:
# Partimos del modelo ya optimizado y reentrenado por el GridSearchCV sobre todo el train,
# así que no hace falta volver a ajustarlo.
modelo_final = grid_lr.best_estimator_

# Recuperamos los nombres de las variables tal y como salen del preprocesamiento,
# para que las gráficas muestren etiquetas legibles y no "x0, x1, x2...".
nombres_variables = modelo_final.named_steps["preprocesamiento"].get_feature_names_out()
nombres_variables = [n.split("__")[-1] for n in nombres_variables]  # quitamos el prefijo del transformador

In [ ]:
# Nivel 1a: Coeficientes de la Regresión Logística.

# Al haber escalado las variables antes, los coeficientes son comparables entre sí:
# un coeficiente positivo empuja hacia "diabetes", uno negativo hacia "no diabetes",
# y su magnitud indica cuánto pesa esa variable en la decisión.

coeficientes = modelo_final.named_steps["clasificador"].coef_[0]

importancia_coef = pd.DataFrame({
    "Variable": nombres_variables,
    "Coeficiente": coeficientes
}).sort_values("Coeficiente", key=abs, ascending=True)

plt.figure(figsize=(8, 5))
colores = ["#c0392b" if c > 0 else "#2874a6" for c in importancia_coef["Coeficiente"]]
plt.barh(importancia_coef["Variable"], importancia_coef["Coeficiente"], color=colores)
plt.axvline(0, color="gray", linewidth=0.8)
plt.xlabel("Peso en la decisión (coeficiente)")
plt.title("Influencia de cada variable en la Regresión Logística")
plt.tight_layout()
plt.savefig("fig13_coeficientes_regresion.png", dpi=300, bbox_inches="tight")
plt.show()

Podemos observar como claramente Glucose, es con diferencia, es la variable con mayor peso de todas, algo que clínicamente tiene todo el sentido, ya que la glucosa en sangre es literalmente el criterio con el que se diagnostica la diabetes.

Detrás aparece el IMC (BMI), que tiene total coherencia debido a la fuerte relación conocida entre obesidad y diabetes tipo 2. Luego, el número de embarazos y el antecedente familiar (DiabetesPedigreeFunction), dos factores de riesgo bien documentados en la literatura.

Todos los coeficientes son positivos, por lo tanto todas las variables empujan hacia el diagnóstico cuando suben.

In [ ]:
# Nivel 1b: Importancia por permutación.

# Mide cuánto empeora el modelo si "estropeamos" una variable barajando sus valores.
# Si al desordenar una variable el rendimiento cae mucho, es que esa variable era importante.
# La ventaja frente a los coeficientes es que no asume relación lineal y se calcula sobre el test.

from sklearn.inspection import permutation_importance

resultado_perm = permutation_importance(
    modelo_final, X_test, y_test,
    scoring="roc_auc", n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1
)

importancia_perm = pd.DataFrame({
    "Variable": X_test.columns,
    "Importancia": resultado_perm.importances_mean,
    "Desviacion": resultado_perm.importances_std
}).sort_values("Importancia", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importancia_perm["Variable"], importancia_perm["Importancia"],
         xerr=importancia_perm["Desviacion"], color="#2874a6", alpha=0.85)
plt.xlabel("Caída del ROC-AUC al permutar la variable")
plt.title("Importancia por permutación (sobre el conjunto de test)")
plt.tight_layout()
plt.savefig("fig14_importancia_permutacion.png", dpi=300, bbox_inches="tight")
plt.show()

Esta gráfica sobre la permutación, cuenta prácticamente la misma historia que la anterior, respaldando que los resultados son coherentes. Cuando dos métodos independientes coinciden en señalar a las mismas variables, el argumento se vuelve mucho más fiable.

Aquí, la Glucose también destaca de forma aplastante, seguida de BMI y Pregnancies, con un ROC-AUC que se desploma cuando desordenamos la glucosa y apenas se inmuta cuando tocamos la presión arterial. Que las dos técnicas apunten al mismo grupo de variables clave refuerza que el modelo no está aprendiendo casualidades, sino relaciones clínicas reales.

### **11.2. Visión local**

In [ ]:
# Nivel 2: SHAP para explicaciones locales y globales.

# SHAP reparte, para cada paciente, cuánto ha aportado cada variable a su predicción concreta.

import shap

# Transformamos el train con el pipeline (sin el clasificador) para que SHAP trabaje
# sobre los datos ya imputados y escalados, que es lo que realmente ve el modelo.
preprocesador_ajustado = modelo_final.named_steps["preprocesamiento"]
X_train_transformado = preprocesador_ajustado.transform(X_train)
X_test_transformado = preprocesador_ajustado.transform(X_test)

clasificador_final = modelo_final.named_steps["clasificador"]

# Para un modelo lineal, LinearExplainer es el explicador adecuado y rápido.
explicador = shap.LinearExplainer(
    clasificador_final,
    shap.maskers.Independent(X_train_transformado, max_samples=X_train_transformado.shape[0]))
shap_values = explicador.shap_values(X_test_transformado)

Una precisión sobre el método. El masker Independent que usa SHAP asume que las variables son independientes entre sí a la hora de repartir las contribuciones. En nuestro caso eso no se cumple del todo, ya que el análisis exploratorio mostró relaciones entre algunas variables, como la de glucosa con insulina o la del IMC con el grosor del pliegue cutáneo. Esto significa que los valores SHAP hay que leerlos como una aproximación razonable al peso de cada variable, no como un reparto exacto. Aun así, al tratarse de un modelo lineal y coincidir sus conclusiones con las de los coeficientes y la importancia por permutación, la interpretación global se mantiene fiable.

In [ ]:
# Resumen global SHAP: muestra qué variables mueven más las predicciones en todo el conjunto de test.
shap.summary_plot(
    shap_values, X_test_transformado,
    feature_names=nombres_variables, show=False
)
plt.title("Resumen SHAP — impacto de cada variable")
plt.tight_layout()
plt.savefig("fig15_shap_resumen.png", dpi=300, bbox_inches="tight")
plt.show()

Aquí se ve algo que las gráficas anteriores no mostraban: no solo qué variables pesan, sino en qué dirección. En la fila de Glucose, los puntos rojos (valores altos de glucosa) se van claramente hacia la derecha, empujando la predicción hacia diabetes, mientras que los azules (glucosa baja) tiran hacia la izquierda. El mismo patrón se repite con el IMC. Es exactamente el comportamiento que esperaríamos de un modelo clínicamente coherente, a más glucosa y más IMC, más riesgo.

In [ ]:
# Explicación de un caso individual: elegimos una paciente concreta del test
# y vemos qué variables la empujaron hacia el diagnóstico de diabetes.

indice_paciente = 0

shap.force_plot(
    explicador.expected_value,
    shap_values[indice_paciente],
    X_test_transformado[indice_paciente],
    feature_names=nombres_variables,
    matplotlib=True, show=False
)
plt.tight_layout()
plt.savefig("fig16_shap_paciente_individual.png", dpi=300, bbox_inches="tight")
plt.show()

En esta paciente concreta, la glucosa alta, los embarazos y la edad empujan con fuerza hacia el diagnóstico de diabetes (la zona roja), mientras que su IMC y el antecedente familiar tiran en sentido contrario (la zona azul). El modelo ofrece un veredicto transparente, ya que muestra el "tira y afloja" entre las variables que lo llevó a esa predicción concreta, y esto es justo lo que necesita un médico para decidir si se fía o si pide una prueba adicional.

In [ ]:
# Función auxiliar para explorar cualquier paciente del test sin tener que reescribir el código.

def explorar_paciente(indice):
    return shap.force_plot(
        explicador.expected_value,
        shap_values[indice],
        X_test_transformado[indice],
        feature_names=nombres_variables,
        matplotlib=True, show=True
    )

explorar_paciente(4)
explorar_paciente(6)

Para no quedarnos con un solo caso, definimos una función que dibuja la explicación de cualquier paciente del test con solo pasarle su índice, sin tener que reescribir el código cada vez. La probamos con dos pacientes que resultan ser opuestas para ver bien el contraste. En la primera, la predicción se va con claridad hacia el lado no diabético, con la glucosa como variable predominante, que empueja con fuerza en esa dirección. En la segunda ocurre lo contrario, el IMC elevado y varios factores más la llevan hacia el diagnóstico de diabetes.

In [ ]:
import numpy as np

# Índices de pacientes diabéticas reales dentro del test
diabeticas = np.where(y_test.values == 1)[0]
print("Posiciones de pacientes diabéticas en el test:", diabeticas[:10])

# Y ahora explicas una de ellas, por ejemplo la primera diabética
explorar_paciente(diabeticas[0])

En vez de ir eligiendo pacientes al azar, aquí primero localizamos dentro del test cuáles son las diabéticas reales y nos quedamos con una de ellas para explicarla en detalle. Tiene más valor porque permite comprobar qué está mirando el modelo cuando se enfrenta a una paciente que sí padece la enfermedad. En este caso, el antecedente familiar y el número de embarazos son los que empujan hacia el diagnóstico de diabetes, mientras que el IMC y la glucosa tiran en sentido contrario. Lo interesante es que aquí el empujón hacia diabetes no viene de la glucosa, como pasaba en la mayoría de casos, sino de los antecedentes y los embarazos, lo que demuestra que el modelo no se guía siempre por la misma variable, sino que pondera el conjunto del perfil de cada paciente. El hecho de señalar a una diabética concreta y desglosar el porqué de su predicción es precisamente el tipo de transparencia obligatoria que debe tener un sistema de apoyo a la decisión clínica.

# ***Conclusiones***

El trabajo arrancó con la carga del dataset de Pima, un conjunto pequeño y con un problema que condicionó buena parte de las decisiones posteriores: varias variables tenían ceros que en realidad eran datos ausentes. En el bloque 1 nos ocupamos de entender las distribuciones y de montar un preprocesamiento honesto, con imputación diferenciada según el tipo de variable y todo encapsulado en un pipeline que se ajusta solo con el train. Esa base, aunque menos vistosa que el modelado, es la que sostiene que los resultados de después sean creíbles y no fruto de una fuga de datos.

En el bloque 2, comparamos cinco algoritmos en igualdad de condiciones mediante validación cruzada estratificada, y dos destacaron por encima del resto, la Regresión Logística y el SVM. Los afinamos con GridSearchCV y llegaron a la evaluación final muy parejos, con un AUC prácticamente idéntico sobre el test. Hasta ahí ninguno de los dos ganaba de forma clara, así que la decisión no podía basarse solo en la métrica global.

El punto de inflexión estuvo en llevar el problema al terreno clínico. Como un falso negativo aquí es mucho más grave que un falso positivo, ajustamos el umbral de decisión buscando un recall del 85%, y lo calculamos sobre el train para no contaminar el test. Ahí la Regresión Logística sacó ventaja, alcanzaba ese mismo recall con un umbral más alto y arrastrando menos falsos positivos, lo que indica que separaba mejor a las pacientes. Ese fue el argumento clave que hizo inclinar la balanza a su favor.

La interpretabilidad terminó de confirmar la elección. Tanto los coeficientes como la importancia por permutación y el análisis SHAP coincidieron en señalar la glucosa y el IMC como los factores de más peso, justo lo que cabía esperar desde el punto de vista médico. Que el modelo elegido sea además el más transparente, y que apoye su decisión en variables clínicamente coherentes, refuerza su potencial como herramienta de apoyo. Conviene precisar que estos resultados se han obtenido sobre un único conjunto de datos, sin validación externa ni prospectiva, de modo que justifican seguir estudiando el modelo, contrastandolo con muchos más datos de pacientes reales y con expertos en bioinformática, antes de tratar de implementarlo en cualquier uso clínico que pueda afectar a la salud de las personas.